# GenoPath E2B — GRPO Fine-Tuning (T4-compatible)

**Task:** Fine-tune Gemma 4 E2B to extract clinical phenotype terms from synthetic patient notes,
aligning output to HPO term names.

**Reward:** F1 overlap between extracted terms and ground-truth HPO term names from GenoPath case generator.

**Hardware target:** Colab T4 (16GB). E2B has 5.1B total params (2.3B effective) due to Per-Layer
Embeddings. At 4-bit + LoRA, expect ~10-12GB VRAM peak — fits T4 with headroom.

**Before running:**
1. Upload `genopath_upload.zip` to your Google Drive
2. Set your HuggingFace token in Cell 2
3. Runtime -> Change runtime type -> T4 GPU -> Run all

In [ ]:
# Cell 1: Install — both unsloth AND unsloth_zoo must come from git HEAD to match
# The ImportError '_unsloth_get_mm_token_id' means unsloth_zoo is stale vs unsloth.
import os
os.environ["UNSLOTH_DISABLE_STATISTICS"] = "1"

!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q "unsloth_zoo @ git+https://github.com/unslothai/unsloth-zoo.git"
!pip install -q trl>=0.9.0 peft accelerate bitsandbytes datasets pydantic rich nest_asyncio
print("Install complete")

## ⚠️ After Cell 1 completes: Runtime → Restart runtime

pip installs stay on disk. After restart, skip Cell 1 and run from Cell 2 onwards.
This avoids the `ImportError`/`OSError` caused by stale cached module versions in the live kernel.

In [ ]:
# Cell 2: Config
HF_TOKEN  = "hf_YOUR_TOKEN_HERE"           # https://huggingface.co/settings/tokens
HF_REPO   = "KrishVenky/genopath-e2b-grpo" # will be created on push

# Official Google model ID (capital E2B — google/gemma-4-2b-it does NOT exist)
# If Unsloth publishes unsloth/gemma-4-E2B-it-bnb-4bit, use that instead for faster load.
BASE_MODEL     = "google/gemma-4-E2B-it"

MAX_SEQ_LENGTH = 1024
LORA_RANK      = 16
MAX_COMPLETION = 600    # MUST be 600 — 300 fills budget with thinking tokens, zero signal
BATCH_SIZE     = 1
GRAD_ACCUM     = 8
LEARNING_RATE  = 2e-5
TRAIN_STEPS    = 150    # ~2-3h on T4; raise to 300 if session allows
SAVE_STEPS     = 50
N_SEEDS        = 60     # seeds 1-60 x 3 task types = 180 training examples

print(f"Base model : {BASE_MODEL}")
print(f"Max steps  : {TRAIN_STEPS}")
print(f"Output     : {HF_REPO}")

In [ ]:
# Cell 3: Mount Google Drive and unzip project files
from google.colab import drive
drive.mount("/content/drive")

import subprocess
ZIP_PATH = "/content/drive/MyDrive/genopath_upload.zip"  # adjust if you put it in a subfolder
subprocess.run(["unzip", "-q", "-o", ZIP_PATH, "-d", "/content/genopath"], check=True)

import sys
sys.path.insert(0, "/content/genopath")
print("Unzipped and sys.path updated")

In [ ]:
# Cell 4: Build GenoPath graph (CPU only, ~8s)
import time
t0 = time.time()
from src.graph.graph import get_graph
graph = get_graph()
print(f"Graph ready in {time.time()-t0:.1f}s — {len(graph.nodes):,} nodes")

In [ ]:
# Cell 5: Generate training dataset from GenoPath cases
# Input: synthetic clinical note built from HPO term names
# Target: same HPO term names as a line-separated list
import random
from src.graph.case_generator import generate_case

SYSTEM = (
    "You are a clinical NLP assistant. Given a patient clinical note, "
    "extract every phenotype or symptom mentioned. "
    "Output one term per line using standard clinical terminology. "
    "No bullet points, no explanations."
)

def make_note(names: list) -> str:
    intro = random.choice([
        "Patient presents with:",
        "Clinical findings include:",
        "Referring physician notes:",
        "Physical examination reveals:",
    ])
    return intro + "\n" + "\n".join(f"- {n}" for n in names)

rows = []
for seed in range(1, N_SEEDS + 1):
    for task in ["monogenic", "oligogenic", "phenotype_mismatch"]:
        try:
            case = generate_case(graph, task, seed=seed)
            rows.append({
                "prompt": [
                    {"role": "system", "content": SYSTEM},
                    {"role": "user",   "content": "Extract phenotypes:\n\n" + make_note(case.patient_phenotype_names)},
                ],
                "ground_truths": case.patient_phenotype_names,  # must match reward_fn parameter name
            })
        except Exception as e:
            print(f"  skip seed={seed} task={task}: {e}")

random.seed(42)
random.shuffle(rows)
print(f"Dataset: {len(rows)} examples")
print("Sample:", rows[0]["ground_truths"])

In [ ]:
# Cell 6: Load Gemma 4 E2B with Unsloth + LoRA
# The installed unsloth (newer) expects symbols from unsloth_zoo.rl_replacements
# that the system unsloth_zoo (older) doesn't have.
# Fix: read exactly what unsloth expects, stub anything missing — all at once.
import ast, unsloth_zoo.rl_replacements as _zoo_rl

_rl_path = "/usr/local/lib/python3.12/dist-packages/unsloth/models/rl_replacements.py"
with open(_rl_path) as _f:
    _tree = ast.parse(_f.read())

_patched = []
for _node in ast.walk(_tree):
    if (isinstance(_node, ast.ImportFrom)
            and _node.module == "unsloth_zoo.rl_replacements"):
        for _alias in _node.names:
            if not hasattr(_zoo_rl, _alias.name):
                setattr(_zoo_rl, _alias.name, lambda *a, **kw: None)
                _patched.append(_alias.name)

if _patched:
    print(f"Patched {len(_patched)} missing symbol(s): {_patched}")
else:
    print("No missing symbols — unsloth_zoo is already compatible.")

from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=None,
    load_in_4bit=True,
    token=HF_TOKEN,
)

# Disable thinking — thinking tokens consume MAX_COMPLETION budget, killing the learning signal.
_orig = tokenizer.apply_chat_template
def _no_think(*a, **kw):
    kw["enable_thinking"] = False
    return _orig(*a, **kw)
tokenizer.apply_chat_template = _no_think

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_RANK,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],
    lora_alpha=LORA_RANK,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)
print("Trainable params:", sum(p.numel() for p in model.parameters() if p.requires_grad))

In [ ]:
# Cell 7: Reward function — phenotype extraction F1

def _f1(pred: list, gt: list) -> float:
    if not gt:
        return 0.0
    p = {t.lower().strip() for t in pred if t.strip()}
    g = {t.lower().strip() for t in gt}
    tp = len(p & g)
    if not tp:
        return 0.0
    prec = tp / len(p) if p else 0.0
    rec  = tp / len(g)
    return 2 * prec * rec / (prec + rec)

def reward_fn(completions, ground_truths, **kwargs):
    # TRL passes completions as List[Dict] (chat format)
    rewards = []
    for comp, gt in zip(completions, ground_truths):
        text = comp[-1]["content"] if isinstance(comp, list) else str(comp)
        extracted = [l.strip("- *\t").strip() for l in text.splitlines() if l.strip()]
        rewards.append(_f1(extracted, gt))
    return rewards

# Smoke test
r = reward_fn(
    [[{"role": "assistant", "content": "Seizures\nHypotonia\nGlobal developmental delay"}]],
    [["Seizures", "Hypotonia", "Global developmental delay", "Microcephaly"]]
)
print(f"Reward smoke test (3/4 correct): {r[0]:.3f}  (expected ~0.857)")

In [ ]:
# Cell 8: Train with GRPO
# Watch reward_std in first 10 steps — if 0.0, thinking mode is on or MAX_COMPLETION too small.
import nest_asyncio
nest_asyncio.apply()

from trl import GRPOConfig, GRPOTrainer

args = GRPOConfig(
    output_dir="/content/genopath-e2b-grpo",
    max_steps=TRAIN_STEPS,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    max_completion_length=MAX_COMPLETION,
    learning_rate=LEARNING_RATE,
    warmup_ratio=0.1,
    logging_steps=5,
    save_steps=SAVE_STEPS,
    fp16=True,
    report_to="none",
    optim="adamw_8bit",
    seed=42,
)

trainer = GRPOTrainer(
    model=model,
    tokenizer=tokenizer,
    args=args,
    train_dataset=rows,
    reward_funcs=reward_fn,
)

stats = trainer.train()
print(f"Done. Runtime: {stats.metrics['train_runtime']:.0f}s")

In [ ]:
# Cell 9: Eval on held-out seeds (61-80, not seen during training)
import traceback, torch
from unsloth import FastLanguageModel as FLM
FLM.for_inference(model)

def extract(note: str, max_new: int = 200) -> list:
    # Gemma4Processor.apply_chat_template requires content as list-of-dicts, not plain strings.
    msgs = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM}]},
        {"role": "user",   "content": [{"type": "text", "text": "Extract phenotypes:\n\n" + note}]},
    ]
    inputs = tokenizer.apply_chat_template(
        msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt"
    )
    # Processor returns BatchEncoding (dict-like); plain tokenizer returns a tensor.
    if hasattr(inputs, "input_ids"):
        input_ids = inputs["input_ids"].to("cuda")
    else:
        input_ids = inputs.to("cuda")
    prompt_len = input_ids.shape[1]
    with torch.no_grad():
        out = model.generate(input_ids=input_ids, max_new_tokens=max_new, use_cache=True)
    sequences = out.sequences if hasattr(out, "sequences") else out
    text = tokenizer.decode(sequences[0][prompt_len:], skip_special_tokens=True)
    return [l.strip("- *\t").strip() for l in text.splitlines() if l.strip()]

eval_scores = []
for seed in range(61, 81):
    try:
        case = generate_case(graph, "monogenic", seed=seed)
        note = make_note(case.patient_phenotype_names)
        preds = extract(note)
        score = _f1(preds, case.patient_phenotype_names)
        eval_scores.append(score)
        print(f"  seed={seed} F1={score:.3f}  pred={preds[:2]}  gt={case.patient_phenotype_names[:2]}")
    except Exception as e:
        traceback.print_exc()
        print(f"  seed={seed} ERROR: {e}")
        break

if eval_scores:
    print(f"\nFine-tuned eval F1  (n={len(eval_scores)}): {sum(eval_scores)/len(eval_scores):.3f}")
    print("Zero-shot baseline typically 0.30-0.50 for HPO alignment")
else:
    print("\nNo eval scores yet — see traceback above.")

In [ ]:
# Cell 10: Push adapter to HF Hub + save results to Drive
import json, shutil

results = {
    "base_model": BASE_MODEL,
    "train_steps": TRAIN_STEPS,
    "eval_f1_mean": round(sum(eval_scores)/len(eval_scores), 4) if eval_scores else None,
    "eval_n": len(eval_scores),
}
with open("/content/training_results.json", "w") as f:
    json.dump(results, f, indent=2)

model.push_to_hub(HF_REPO, token=HF_TOKEN)
tokenizer.push_to_hub(HF_REPO, token=HF_TOKEN)
print(f"Adapter live at https://huggingface.co/{HF_REPO}")

shutil.copy("/content/training_results.json",
            "/content/drive/MyDrive/genopath_training_results.json")
print("Results backed up to Drive")